## Camada Silver — `ecommerce_produtos`

Este notebook lê o micro-lote novo da camada **Bronze física** (`squad1/bronze/ecommerce_produtos`), aplica as 10 regras de qualidade (5 técnicas + 5 de negócio), gera logs de DQ e grava a Silver em `append` no caminho físico `squad1/silver/ecommerce_produtos`, **com particionamento físico Hive-style**.

### O que foi adaptado aqui

1. **Leitura da Bronze**: troquei `spark.table(TABELA_BRONZE)` por leitura física via SDK `deltalake`, do caminho `squad1/bronze/ecommerce_produtos`.
2. **Idempotência**: a verificação de `bronze_source_file` já processados passou a consultar a tabela Delta física da Silver (`squad1/silver/ecommerce_produtos`).
3. **Gravação da Silver**: mecanismo `gravar_delta` (SDK `deltalake`) — bypass do Databricks Serverless.
4. **Particionamento físico Hive-style da Silver**: `silver_processed_year/month/day/hour`.
5. **DQ Logs**: gravados em `squad1/dq_monitoring_logs` (raiz do container), sem particionamento.
6. **Referências físicas**: categorias lidas de `squad1/bronze/ecommerce_categorias`; itens de pedido lidos de `squad1/bronze/ecommerce_itens_pedido` — ambos via SDK `deltalake`.
7. **Sem dependência de notebook externo**: todas as funções auxiliares definidas dentro deste notebook.


## 1. Instalação do SDK `deltalake` (caso não esteja disponível no cluster)

In [0]:
%pip install -q "deltalake==0.15.3" "pyarrow==14.0.1" python-dotenv azure-identity azure-storage-file-datalake

## 2. Imports, parâmetros e credenciais

In [0]:
# Imports e parâmetros

import os
import uuid
import pandas as pd
import pyarrow as pa
from functools import reduce
from datetime import datetime, timezone
from dotenv import load_dotenv

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

from deltalake import DeltaTable
from deltalake.writer import write_deltalake

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# --- Origem física (Bronze, Delta) ---
CONTAINER_SQUAD1 = "squad1"
CAMADA_ORIGEM    = "bronze"
ENTIDADE_ORIGEM  = "ecommerce_produtos"

# --- Destino físico (Silver, Delta) ---
CAMADA_DESTINO   = "silver"
ENTIDADE_DESTINO = "ecommerce_produtos"

# --- Destino físico dos logs de DQ ---
CAMADA_DQ   = ""
ENTIDADE_DQ = "dq_monitoring_logs"

# --- Referências físicas para as regras ---
CAMADA_CATEGORIAS_ORIGEM   = "bronze"
ENTIDADE_CATEGORIAS_ORIGEM = "ecommerce_categorias"

CAMADA_ITENS_ORIGEM   = "bronze"
ENTIDADE_ITENS_ORIGEM = "ecommerce_itens_pedido"

# --- Particionamento físico Hive-style ---
PARTICIONAR_SILVER  = True
PARTICIONAR_DQ_LOGS = False

RUN_ID = str(uuid.uuid4())

print("Origem física (Bronze)  :", f"{CONTAINER_SQUAD1}/{CAMADA_ORIGEM}/{ENTIDADE_ORIGEM}")
print("Destino físico (Silver) :", f"{CONTAINER_SQUAD1}/{CAMADA_DESTINO}/{ENTIDADE_DESTINO}")
print("Destino físico (DQ Logs):", f"{CONTAINER_SQUAD1}/{ENTIDADE_DQ}")
print("Referência (Categorias) :", f"{CONTAINER_SQUAD1}/{CAMADA_CATEGORIAS_ORIGEM}/{ENTIDADE_CATEGORIAS_ORIGEM}")
print("Referência (Itens)      :", f"{CONTAINER_SQUAD1}/{CAMADA_ITENS_ORIGEM}/{ENTIDADE_ITENS_ORIGEM}")
print("RUN_ID                  :", RUN_ID)

# Credenciais do Service Principal (.env):
load_dotenv("/Workspace/Users/soaress.elias@gmail.com/merca-data-platform-categorias/.env")

CLIENT_ID            = os.getenv("ADLS_CLIENT_ID")
TENANT_ID            = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET        = os.getenv("ADLS_CLIENT_SECRET")
STORAGE_ACCOUNT_NAME = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")

STORAGE_OPTIONS = {
    "account_name":  STORAGE_ACCOUNT_NAME,
    "client_id":     CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "tenant_id":     TENANT_ID,
}

credential = ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)

service_client = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    credential=credential
)
file_system_squad1 = service_client.get_file_system_client(file_system=CONTAINER_SQUAD1)

print("✅ Credenciais carregadas com sucesso.")

## 3. Funções auxiliares

`get_delta_path`, `delta_existe` e `gravar_delta`. `gravar_delta` foi **generalizada** para reconhecer tanto as colunas de partição do Bronze (`bronze_ingest_*`) quanto as da Silver (`silver_processed_*`). `ler_delta` e `obter_arquivos_ja_processados` leem tabelas Delta físicas em vez de tabelas do Unity Catalog.


In [0]:
from io import BytesIO
import json
import uuid as uuid_lib

def get_delta_path(camada: str, tabela: str, storage_opts: dict) -> str:
    conta = storage_opts.get("account_name")
    if not camada:
        return f"abfss://{CONTAINER_SQUAD1}@{conta}.dfs.core.windows.net/{tabela}"
    return f"abfss://{CONTAINER_SQUAD1}@{conta}.dfs.core.windows.net/{camada}/{tabela}"


def _pasta_fisica(camada: str, tabela: str) -> str:
    """Retorna o caminho relativo dentro do container squad1."""
    return f"{camada}/{tabela}" if camada else tabela


def delta_existe(camada: str, tabela: str, storage_opts: dict) -> bool:
    """Verifica se existe _delta_log no caminho físico via DataLakeServiceClient."""
    try:
        pasta = f"{_pasta_fisica(camada, tabela)}/_delta_log"
        paths = list(file_system_squad1.get_paths(path=pasta, recursive=False))
        return len(paths) > 0
    except Exception:
        return False


def ler_delta(camada: str, tabela: str, storage_opts: dict):
    """Lê todos os arquivos Parquet da tabela Delta via DataLakeServiceClient
    e retorna um DataFrame PySpark."""
    if not delta_existe(camada, tabela, storage_opts):
        raise Exception(f"Tabela Delta física não encontrada: {camada}/{tabela}")

    pasta_raiz = _pasta_fisica(camada, tabela)
    todos_os_paths = list(file_system_squad1.get_paths(path=pasta_raiz, recursive=True))
    arquivos_parquet = [
        p.name for p in todos_os_paths
        if not p.is_directory and p.name.endswith(".parquet")
    ]

    if not arquivos_parquet:
        raise Exception(f"Nenhum arquivo Parquet encontrado em {pasta_raiz}")

    print(f"Lendo {len(arquivos_parquet)} arquivo(s) Parquet de {pasta_raiz}...")

    dfs = []
    for arquivo in arquivos_parquet:
        file_client = file_system_squad1.get_file_client(arquivo)
        conteudo = file_client.download_file().readall()
        pdf = pd.read_parquet(BytesIO(conteudo))
        dfs.append(pdf)

    pdf_total = pd.concat(dfs, ignore_index=True)

    for col_name in pdf_total.columns:
        if pd.api.types.is_datetime64_any_dtype(pdf_total[col_name]):
            try:
                pdf_total[col_name] = pdf_total[col_name].dt.tz_localize(None)
            except TypeError:
                pdf_total[col_name] = pdf_total[col_name].dt.tz_convert(None)

    return spark.createDataFrame(pdf_total)


def _proximo_version_delta(pasta_raiz: str) -> int:
    """Retorna o próximo número de versão do _delta_log."""
    pasta_log = f"{pasta_raiz}/_delta_log"
    try:
        paths = list(file_system_squad1.get_paths(path=pasta_log, recursive=False))
        jsons = [p.name for p in paths if p.name.endswith(".json")]
        if not jsons:
            return 0
        versoes = [int(os.path.basename(p).replace(".json", "")) for p in jsons]
        return max(versoes) + 1
    except Exception:
        return 0


def _upload_bytes(caminho_relativo: str, conteudo: bytes):
    """Faz upload de bytes para um caminho no container squad1."""
    file_client = file_system_squad1.get_file_client(caminho_relativo)
    file_client.create_file()
    file_client.append_data(conteudo, offset=0, length=len(conteudo))
    file_client.flush_data(len(conteudo))


def gravar_delta(df, camada: str, tabela: str, storage_opts: dict, mode: str = "append", particionar: bool = True) -> bool:
    """Grava um DataFrame Spark como Delta físico via DataLakeServiceClient.
    Escreve os arquivos Parquet e o _delta_log manualmente — sem write_deltalake,
    sem Rust, sem dependência de autenticação OAuth no sandbox do Serverless."""
    pasta_raiz = _pasta_fisica(camada, tabela)
    ja_existe = delta_existe(camada, tabela, storage_opts)
    modo_real = mode if ja_existe else "overwrite"

    try:
        pdf = df.toPandas()

        # Normaliza datetimes
        for col_name in pdf.columns:
            if pd.api.types.is_datetime64_any_dtype(pdf[col_name]):
                try:
                    pdf[col_name] = pdf[col_name].dt.tz_localize(None)
                except TypeError:
                    pdf[col_name] = pdf[col_name].dt.tz_convert(None)

        # Define colunas de partição
        partition_cols = []
        if particionar:
            colunas_particao_por_camada = {
                "bronze": ["bronze_ingest_year", "bronze_ingest_month", "bronze_ingest_day", "bronze_ingest_hour"],
                "silver": ["silver_processed_year", "silver_processed_month", "silver_processed_day", "silver_processed_hour"],
            }
            possiveis = colunas_particao_por_camada.get(camada, [])
            partition_cols = [c for c in possiveis if c in pdf.columns.tolist()]

        # Obtém próxima versão do delta_log
        versao = _proximo_version_delta(pasta_raiz)

        # Grava os arquivos Parquet (particionado ou não)
        arquivos_gravados = []

        if partition_cols:
            grupos = pdf.groupby(partition_cols)
            for chave, grupo in grupos:
                # Monta o caminho da partição hive-style
                if isinstance(chave, tuple):
                    partes = [f"{c}={v}" for c, v in zip(partition_cols, chave)]
                else:
                    partes = [f"{partition_cols[0]}={chave}"]
                subpasta = "/".join(partes)
                nome_arquivo = f"part-{str(uuid_lib.uuid4())[:8]}.snappy.parquet"
                caminho_parquet = f"{pasta_raiz}/{subpasta}/{nome_arquivo}"

                buffer = BytesIO()
                grupo_arrow = pa.Table.from_pandas(grupo, preserve_index=False)
                import pyarrow.parquet as pq
                pq.write_table(grupo_arrow, buffer, compression="snappy")
                _upload_bytes(caminho_parquet, buffer.getvalue())
                arquivos_gravados.append((caminho_parquet, subpasta, len(grupo)))
        else:
            nome_arquivo = f"part-{str(uuid_lib.uuid4())[:8]}.snappy.parquet"
            caminho_parquet = f"{pasta_raiz}/{nome_arquivo}"
            buffer = BytesIO()
            arrow_table = pa.Table.from_pandas(pdf, preserve_index=False)
            import pyarrow.parquet as pq
            pq.write_table(arrow_table, buffer, compression="snappy")
            _upload_bytes(caminho_parquet, buffer.getvalue())
            arquivos_gravados.append((caminho_parquet, None, len(pdf)))

        # Monta o schema Arrow para o _delta_log
        arrow_schema = pa.Table.from_pandas(pdf.head(0), preserve_index=False).schema

        def arrow_type_to_delta(t):
            import pyarrow as pa
            if pa.types.is_string(t) or pa.types.is_large_string(t):
                return "string"
            elif pa.types.is_int32(t):
                return "integer"
            elif pa.types.is_int64(t):
                return "long"
            elif pa.types.is_float32(t):
                return "float"
            elif pa.types.is_float64(t):
                return "double"
            elif pa.types.is_boolean(t):
                return "boolean"
            elif pa.types.is_timestamp(t):
                return "timestamp"
            elif pa.types.is_date32(t):
                return "date"
            else:
                return "string"

        delta_schema_fields = [
            {"name": field.name, "type": arrow_type_to_delta(field.type), "nullable": True, "metadata": {}}
            for field in arrow_schema
        ]
        delta_schema = {
            "type": "struct",
            "fields": delta_schema_fields
        }

        # Grava o _delta_log
        if versao == 0:
            # Primeira gravação: commitInfo + metaData + protocol + add
            commit_info = {
                "commitInfo": {
                    "timestamp": int(datetime.now(timezone.utc).timestamp() * 1000),
                    "operation": "WRITE",
                    "operationParameters": {"mode": "Overwrite", "partitionBy": json.dumps(partition_cols)},
                    "isBlindAppend": False
                }
            }
            metadata = {
                "metaData": {
                    "id": str(uuid_lib.uuid4()),
                    "format": {"provider": "parquet", "options": {}},
                    "schemaString": json.dumps(delta_schema),
                    "partitionColumns": partition_cols,
                    "configuration": {},
                    "createdTime": int(datetime.now(timezone.utc).timestamp() * 1000)
                }
            }
            protocol = {
                "protocol": {"minReaderVersion": 1, "minWriterVersion": 2}
            }
            linhas_log = [commit_info, metadata, protocol]
        else:
            # Append: só commitInfo + add
            linhas_log = [{
                "commitInfo": {
                    "timestamp": int(datetime.now(timezone.utc).timestamp() * 1000),
                    "operation": "WRITE",
                    "operationParameters": {"mode": "Append", "partitionBy": json.dumps(partition_cols)},
                    "isBlindAppend": True
                }
            }]

        # Adiciona entradas "add" para cada arquivo gravado
        for caminho_parquet, subpasta, num_linhas in arquivos_gravados:
            nome_relativo = caminho_parquet.replace(f"{pasta_raiz}/", "")
            partition_values = {}
            if subpasta:
                for parte in subpasta.split("/"):
                    k, v = parte.split("=")
                    partition_values[k] = v
            add_entry = {
                "add": {
                    "path": nome_relativo,
                    "partitionValues": partition_values,
                    "size": 0,
                    "modificationTime": int(datetime.now(timezone.utc).timestamp() * 1000),
                    "dataChange": True
                }
            }
            linhas_log.append(add_entry)

        # Serializa e faz upload do arquivo de log
        nome_log = f"{str(versao).zfill(20)}.json"
        caminho_log = f"{pasta_raiz}/_delta_log/{nome_log}"
        conteudo_log = "\n".join(json.dumps(linha) for linha in linhas_log).encode("utf-8")
        _upload_bytes(caminho_log, conteudo_log)

        print(f"[Sucesso] Gravado fisicamente em: {pasta_raiz} | Linhas: {len(pdf)} | Versão Delta: {versao}")
        return True

    except Exception as e:
        import traceback
        print(f"[Erro] Falha ao gravar {pasta_raiz}: {str(e)}")
        traceback.print_exc()
        return False


def obter_arquivos_ja_processados(camada: str, tabela: str, storage_opts: dict, coluna_arquivo: str = "bronze_source_file") -> set:
    """Lê a coluna de controle diretamente dos Parquet físicos via DataLakeServiceClient."""
    if not delta_existe(camada, tabela, storage_opts):
        return set()
    try:
        pasta_raiz = _pasta_fisica(camada, tabela)
        todos_os_paths = list(file_system_squad1.get_paths(path=pasta_raiz, recursive=True))
        arquivos_parquet = [
            p.name for p in todos_os_paths
            if not p.is_directory and p.name.endswith(".parquet")
        ]
        valores = set()
        for arquivo in arquivos_parquet:
            file_client = file_system_squad1.get_file_client(arquivo)
            conteudo = file_client.download_file().readall()
            pdf = pd.read_parquet(BytesIO(conteudo), columns=[coluna_arquivo])
            valores.update(pdf[coluna_arquivo].dropna().unique().tolist())
        return valores
    except Exception as e:
        print(f"[Aviso] Não foi possível ler arquivos já processados: {e}")
        return set()


def filtrar_micro_lote_novo(df_bronze, arquivos_processados: set, coluna_arquivo: str = "bronze_source_file"):
    if not arquivos_processados:
        print("Nenhum registro encontrado na Silver física ainda. Processando lote completo.")
        return df_bronze
    df_processados = spark.createDataFrame([(a,) for a in arquivos_processados], [coluna_arquivo])
    return df_bronze.join(df_processados, on=coluna_arquivo, how="left_anti")

## 4. Ler Bronze física e filtrar o micro-lote novo (idempotência por `bronze_source_file`)

In [0]:
df_bronze = ler_delta(camada=CAMADA_ORIGEM, tabela=ENTIDADE_ORIGEM, storage_opts=STORAGE_OPTIONS)

arquivos_ja_processados = obter_arquivos_ja_processados(
    camada=CAMADA_DESTINO,
    tabela=ENTIDADE_DESTINO,
    storage_opts=STORAGE_OPTIONS,
    coluna_arquivo="bronze_source_file"
)

df_micro_lote = filtrar_micro_lote_novo(
    df_bronze=df_bronze,
    arquivos_processados=arquivos_ja_processados,
    coluna_arquivo="bronze_source_file"
)

qtd = df_micro_lote.count()
print("Registros novos para processar:", qtd)

if qtd == 0:
    dbutils.notebook.exit("Nenhum arquivo novo para processar na Silver.")

display(
    df_micro_lote
    .select("bronze_source_file")
    .dropDuplicates()
    .orderBy("bronze_source_file")
)


## 5. Padronização + Preparação de referências

Todas as referências externas (categorias e itens de pedido) são lidas pelo mesmo mecanismo físico
(`ler_delta` via SDK `deltalake`) — sem Unity Catalog.


In [0]:
# --- Padronização dos campos principais ---
df_base = (
    df_micro_lote
    .withColumn("sku_norm",         F.trim(F.col("sku").cast("string")))
    .withColumn("nome_produto_norm", F.trim(F.col("nome_produto")))
    .withColumn("nome_marca_norm",   F.trim(F.col("nome_marca")))
    .withColumn("id_categoria_str",  F.trim(F.col("id_categoria").cast("string")))
    .withColumn("preco_lista_num",   F.col("preco_lista").cast("double"))
    .withColumn("is_ativo_bool",     F.col("is_ativo").cast("boolean"))
)

# -------------------------------------------------------------------
# Referência de Categorias — Regra 4 (id_categoria deve existir) e
# Regra 10 (produto deve estar em subcategoria, id_categoria_pai IS NOT NULL)
# Lida de squad1/bronze/ecommerce_categorias via SDK deltalake.
# -------------------------------------------------------------------
if delta_existe(camada=CAMADA_CATEGORIAS_ORIGEM, tabela=ENTIDADE_CATEGORIAS_ORIGEM, storage_opts=STORAGE_OPTIONS):
    df_categorias_bronze = ler_delta(
        camada=CAMADA_CATEGORIAS_ORIGEM,
        tabela=ENTIDADE_CATEGORIAS_ORIGEM,
        storage_opts=STORAGE_OPTIONS
    )
    # Para R4: conjunto de IDs de categoria existentes
    df_categorias_ref = (
        df_categorias_bronze
        .select(F.col("id_categoria").cast("string").alias("id_categoria_valida"))
        .where(F.col("id_categoria_valida").isNotNull())
        .distinct()
    )
    # Para R10: id_categoria_pai de cada categoria (NULL = raiz, NOT NULL = subcategoria)
    df_categorias_com_pai = (
        df_categorias_bronze
        .select(
            F.col("id_categoria").cast("string").alias("id_categoria_str"),
            F.col("id_categoria_pai")
        )
        .distinct()
    )
    print(f"Referência de categorias carregada de squad1/{CAMADA_CATEGORIAS_ORIGEM}/{ENTIDADE_CATEGORIAS_ORIGEM}.")
else:
    print(
        f"[Aviso] Tabela Delta física squad1/{CAMADA_CATEGORIAS_ORIGEM}/{ENTIDADE_CATEGORIAS_ORIGEM} "
        "ainda não encontrada. Regras 4 e 10 tratarão todos os produtos como inválidos."
    )
    df_categorias_ref = spark.createDataFrame([], "id_categoria_valida string")
    df_categorias_com_pai = spark.createDataFrame([], "id_categoria_str string, id_categoria_pai string")

# -------------------------------------------------------------------
# Referência de Vendas — Regra 9
# (produtos ativos devem ter ao menos 1 venda nos últimos 90 dias)
# Lida de squad1/bronze/ecommerce_itens_pedido via SDK deltalake.
# Se a tabela ainda não existir, trata conservadoramente: todos os
# produtos ativos serão marcados como "sem venda recente".
# -------------------------------------------------------------------
if delta_existe(camada=CAMADA_ITENS_ORIGEM, tabela=ENTIDADE_ITENS_ORIGEM, storage_opts=STORAGE_OPTIONS):
    df_itens_bronze = ler_delta(
        camada=CAMADA_ITENS_ORIGEM,
        tabela=ENTIDADE_ITENS_ORIGEM,
        storage_opts=STORAGE_OPTIONS
    )
    # Filtra itens dos últimos 90 dias (usando bronze_ingested_at como proxy de dt_pedido,
    # caso dt_pedido não exista — ajustar o nome da coluna conforme o schema real).
    coluna_data_pedido = "dt_pedido" if "dt_pedido" in df_itens_bronze.columns else "bronze_ingested_at"
    df_vendas_ref = (
        df_itens_bronze
        .filter(F.col(coluna_data_pedido) >= F.current_date() - F.expr("INTERVAL 90 DAYS"))
        .select(F.col("sku").cast("string").alias("sku_norm"))
        .where(F.col("sku_norm").isNotNull())
        .distinct()
        .withColumn("tem_venda_recente", F.lit(True))
    )
    print(f"Referência de vendas recentes carregada de squad1/{CAMADA_ITENS_ORIGEM}/{ENTIDADE_ITENS_ORIGEM}.")
else:
    print(
        f"[Aviso] Tabela Delta física squad1/{CAMADA_ITENS_ORIGEM}/{ENTIDADE_ITENS_ORIGEM} "
        "ainda não encontrada. Regra 9 tratará todos os produtos ativos como sem venda recente."
    )
    df_vendas_ref = spark.createDataFrame([], "sku_norm string, tem_venda_recente boolean")


## 6. Aplicar as 10 Regras

### Correções aplicadas em relação à versão anterior

| Regra | Bug original | Correção |
|-------|-------------|----------|
| **R4** | `id_categoria_str.isNotNull() & id_categoria_str.isNull()` — sempre `False`, regra nunca disparava | Left join com `df_categorias_ref`; falha quando `id_categoria_valida` é NULL após o join (categoria não encontrada) |
| **R5** | `~F.col("unidade_medida").isin(...)` retorna `null` para valores NULL — nulos não eram detectados | `F.col("unidade_medida").isNull() \| ~F.col("unidade_medida").isin(...)` |
| **R9** | Referenciava `TABELAS_SILVER` (variável inexistente) + Unity Catalog | Lê `squad1/bronze/ecommerce_itens_pedido` via SDK deltalake; tratamento conservador se não existir |
| **R10** | Usava `id_categoria_pai` diretamente do Bronze (sem join com categorias) | Join correto com `df_categorias_com_pai` para obter o `id_categoria_pai` da tabela de categorias |


In [0]:
# Janela para detectar duplicatas de SKU dentro do lote
w_sku = Window.partitionBy("sku_norm")

df_regras = (
    df_base

    # Contagem de ocorrências do mesmo SKU no lote (Regra 1)
    .withColumn("qtd_sku_no_lote", F.count("*").over(w_sku))

    # ── Join com tabela de categorias ────────────────────────────────────────
    # Left join para obter id_categoria_pai (usado em R10).
    # A presença/ausência de id_categoria_valida após este join serve para R4.
    .join(df_categorias_com_pai, on="id_categoria_str", how="left")

    # Left join com df_categorias_ref para R4 (detecta IDs inválidos)
    .join(
        df_categorias_ref,
        F.col("id_categoria_str") == F.col("id_categoria_valida"),
        how="left"
    )

    # ── Join com vendas recentes (Regra 9) ───────────────────────────────────
    .join(df_vendas_ref, on="sku_norm", how="left")
    .withColumn("tem_venda_recente", F.coalesce(F.col("tem_venda_recente"), F.lit(False)))

    # ==================== REGRAS TÉCNICAS ====================

    # R1: SKU não pode ser nulo nem duplicado no lote
    # (PK da tabela; duplicata causa problemas em itens_pedido e estoque)
    .withColumn("r1_sku_falhou",
                F.col("sku_norm").isNull() |
                (F.col("sku_norm") == "") |
                (F.col("qtd_sku_no_lote") > 1))

    # R2: preco_lista deve ser > 0
    # (produto com preço zero ou negativo não pode ser comercializado)
    .withColumn("r2_preco_falhou",
                F.col("preco_lista_num").isNull() | (F.col("preco_lista_num") <= 0))

    # R3: nome_produto não pode ser nulo ou string vazia
    # (exibido no front-end e na NF; NULL quebra a experiência do usuário)
    .withColumn("r3_nome_falhou",
                F.col("nome_produto_norm").isNull() |
                (F.col("nome_produto_norm") == ""))

    # R4: id_categoria deve existir na tabela ecommerce_categorias (FK)
    # CORRIGIDO: a versão anterior tinha a lógica invertida e nunca disparava.
    # Agora: falha quando id_categoria_str é não-nulo mas não foi encontrado
    # na referência de categorias (id_categoria_valida == null após left join).
    .withColumn("r4_categoria_invalida_falhou",
                F.col("id_categoria_str").isNotNull() &
                F.col("id_categoria_valida").isNull())

    # R5: unidade_medida deve estar na lista de valores permitidos
    # CORRIGIDO: a versão anterior usava ~isin(), que retorna null para valores
    # NULL — nulos escapavam sem serem detectados.
    # Valores válidos: 'un', 'kg', 'g', 'L', 'ml'
    .withColumn("r5_unidade_falhou",
                F.col("unidade_medida").isNull() |
                ~F.col("unidade_medida").isin(["un", "kg", "g", "L", "ml"]))

    # R6: is_ativo deve ser booleano (TRUE/FALSE) e não pode ser nulo
    # (campo de controle de vitrine; NULL impede lógica de filtro)
    .withColumn("r6_is_ativo_falhou",
                F.col("is_ativo_bool").isNull())

    # ==================== REGRAS DE NEGÓCIO ====================

    # R7: preco_lista deve ser <= R$ 5.000 (limite de categoria seca)
    # (acima desse valor indica erro de precificação para produtos secos)
    .withColumn("r7_preco_alto_falhou",
                F.col("preco_lista_num").isNotNull() & (F.col("preco_lista_num") > 5000))

    # R8: nome_marca não pode ser nulo ou string vazia
    # (obrigatório para rastreabilidade e relatório de sell-out por fornecedor)
    .withColumn("r8_marca_falhou",
                F.col("nome_marca_norm").isNull() |
                (F.col("nome_marca_norm") == ""))

    # R9: Produtos ativos (is_ativo=TRUE) devem ter ao menos 1 venda nos últimos 90 dias
    # CORRIGIDO: versão anterior referenciava TABELAS_SILVER (variável inexistente)
    # e lia via Unity Catalog. Agora lê squad1/bronze/ecommerce_itens_pedido via
    # SDK deltalake; se a tabela não existir, trata conservadoramente.
    .withColumn("r9_sem_venda_falhou",
                (F.col("is_ativo_bool") == True) & (~F.col("tem_venda_recente")))

    # R10: Cada SKU deve pertencer a uma subcategoria (id_categoria_pai IS NOT NULL)
    # CORRIGIDO: a versão anterior usava id_categoria_pai do próprio registro Bronze
    # (sem join com categorias). Agora usa id_categoria_pai obtido do join com
    # df_categorias_com_pai, que reflete a estrutura real da hierarquia de categorias.
    .withColumn("r10_sem_subcategoria_falhou",
                F.col("id_categoria_pai").isNull())

    # Flag final: linha válida somente se todas as regras passaram
    .withColumn("silver_linha_valida",
                ~(
                    F.col("r1_sku_falhou")              |
                    F.col("r2_preco_falhou")             |
                    F.col("r3_nome_falhou")              |
                    F.col("r4_categoria_invalida_falhou")|
                    F.col("r5_unidade_falhou")           |
                    F.col("r6_is_ativo_falhou")          |
                    F.col("r7_preco_alto_falhou")        |
                    F.col("r8_marca_falhou")             |
                    F.col("r9_sem_venda_falhou")         |
                    F.col("r10_sem_subcategoria_falhou")
                ))

    .withColumn("silver_processed_at", F.current_timestamp())

    # Colunas de particionamento físico Hive-style da Silver
    .withColumn("silver_processed_year",  F.year(F.col("silver_processed_at")))
    .withColumn("silver_processed_month", F.month(F.col("silver_processed_at")))
    .withColumn("silver_processed_day",   F.dayofmonth(F.col("silver_processed_at")))
    .withColumn("silver_processed_hour",  F.hour(F.col("silver_processed_at")))

    # Limpeza de colunas auxiliares de join
    .drop("id_categoria_valida", "qtd_sku_no_lote")
)

print("✅ Regras aplicadas com sucesso para ecommerce_produtos (versão corrigida para Serverless)")
display(df_regras.select(
    "sku", "nome_produto", "id_categoria", "preco_lista", "unidade_medida",
    "silver_linha_valida",
    "r1_sku_falhou", "r2_preco_falhou", "r4_categoria_invalida_falhou",
    "r5_unidade_falhou", "r9_sem_venda_falhou", "r10_sem_subcategoria_falhou"
).limit(15))


## 7. Resumo das regras

In [0]:
flags = [c for c in df_regras.columns if c.startswith("r") and c.endswith("_falhou")]

display(df_regras.groupBy("silver_linha_valida").count())
display(df_regras.agg(*[F.sum(F.when(F.col(c), 1).otherwise(0)).alias(c) for c in flags]))


## 8. Gerar DQ Logs

In [0]:
regras_config = [
    ("R1 - sku não nulo/único",         "r1_sku_falhou",               "Critica"),
    ("R2 - preco_lista > 0",            "r2_preco_falhou",             "Critica"),
    ("R3 - nome_produto obrigatório",   "r3_nome_falhou",              "Critica"),
    ("R4 - id_categoria válido (FK)",   "r4_categoria_invalida_falhou","Critica"),
    ("R5 - unidade_medida válida",      "r5_unidade_falhou",           "Critica"),
    ("R6 - is_ativo booleano não nulo", "r6_is_ativo_falhou",          "Critica"),
    ("R7 - preco_lista <= 5000",        "r7_preco_alto_falhou",        "Aviso"),
    ("R8 - nome_marca obrigatório",     "r8_marca_falhou",             "Critica"),
    ("R9 - ativo com venda recente",    "r9_sem_venda_falhou",         "Aviso"),
    ("R10 - pertence a subcategoria",   "r10_sem_subcategoria_falhou", "Aviso"),
]

def criar_log_regra(df, nome_regra, coluna_flag, severidade):
    return (
        df.groupBy("bronze_source_file")
        .agg(
            F.count("*").cast("int").alias("qtd_registros_total"),
            F.sum(F.when(F.col(coluna_flag), 1).otherwise(0)).cast("int").alias("qtd_registros_falhos")
        )
        .withColumn("run_id",             F.lit(RUN_ID))
        .withColumn("tabela",             F.lit("silver_ecommerce_produtos"))
        .withColumn("regra",              F.lit(nome_regra))
        .withColumn("status",             F.when(F.col("qtd_registros_falhos") > 0, "FAIL").otherwise("PASS"))
        .withColumn("severidade",         F.lit(severidade))
        .withColumn("timestamp_execucao", F.current_timestamp())
        .withColumnRenamed("bronze_source_file", "arquivo_origem")
        .select(
            "run_id", "tabela", "regra", "status", "severidade",
            "qtd_registros_falhos", "qtd_registros_total", "timestamp_execucao", "arquivo_origem"
        )
    )

logs = [criar_log_regra(df_regras, nome, flag, sev) for nome, flag, sev in regras_config]
df_dq_logs = reduce(lambda a, b: a.unionByName(b), logs)

display(df_dq_logs)


## 9. Gravar Silver física (Delta particionado) + DQ Logs físicos

Ambas as gravações usam o mesmo mecanismo `gravar_delta` (SDK `deltalake`) —
sem `saveAsTable`, sem dependência do Unity Catalog.

- **Silver**: `squad1/silver/ecommerce_produtos` — particionada por `silver_processed_year/month/day/hour`.
- **DQ Logs**: `squad1/dq_monitoring_logs` — raiz do container, sem particionamento (tabela de controle/auditoria).


In [0]:
# 1. Gravar a Silver física, particionada por silver_processed_year/month/day/hour
sucesso_silver = gravar_delta(
    df=df_regras,
    camada=CAMADA_DESTINO,
    tabela=ENTIDADE_DESTINO,
    storage_opts=STORAGE_OPTIONS,
    mode="append",
    particionar=PARTICIONAR_SILVER
)

if sucesso_silver:
    print("✅ Silver gravada com sucesso (Delta físico particionado via deltalake SDK)!")
else:
    raise Exception("Falha ao gravar a camada Silver. Veja a mensagem de erro acima.")

# 2. Gravar os DQ Logs físicos (sem particionamento — tabela de controle/auditoria)
# Gravados em squad1/dq_monitoring_logs (raiz do container squad1)
sucesso_dq = gravar_delta(
    df=df_dq_logs,
    camada=CAMADA_DQ,
    tabela=ENTIDADE_DQ,
    storage_opts=STORAGE_OPTIONS,
    mode="append",
    particionar=PARTICIONAR_DQ_LOGS
)

if sucesso_dq:
    print("✅ DQ Logs gravados com sucesso em squad1/dq_monitoring_logs!")
else:
    print("⚠️ Falha ao gravar os DQ Logs. Veja a mensagem de erro acima.")


## 10. Validação final (lógica + física)

In [0]:
print("=" * 80)
print("SILVER CONCLUÍDA - ecommerce_produtos")
print("RUN_ID:", RUN_ID)
print("Registros processados:", df_regras.count())
print("Logs DQ gerados:", df_dq_logs.count())
print("Destino físico Silver:", f"{CONTAINER_SQUAD1}/{CAMADA_DESTINO}/{ENTIDADE_DESTINO}")
print("=" * 80)

display(df_regras.select("bronze_source_file").dropDuplicates().orderBy("bronze_source_file"))
display(df_dq_logs.groupBy("status", "severidade").agg(F.sum("qtd_registros_falhos").alias("falhas")))

# --- Validação física do caminho squad1/silver/ecommerce_produtos ---
pasta_silver = f"{CAMADA_DESTINO}/{ENTIDADE_DESTINO}"
print(f"\nArquivos físicos dentro de squad1/{pasta_silver}:")
try:
    for item in file_system_squad1.get_paths(path=pasta_silver, recursive=True):
        tipo = "📁" if item.is_directory else "📄"
        print(f"{tipo} {item.name}")
except Exception as e:
    print("Não foi possível listar o destino Silver:", e)

# --- Validação Delta: lê _delta_log via DataLakeServiceClient (sem DeltaTable/Rust) ---
print("\nValidação Delta + partições físicas (Hive-style):")
if delta_existe(camada=CAMADA_DESTINO, tabela=ENTIDADE_DESTINO, storage_opts=STORAGE_OPTIONS):
    print(f"✅ Tabela Delta física validada em squad1/{CAMADA_DESTINO}/{ENTIDADE_DESTINO}.")

    # Lista partições físicas lendo as subpastas (sem DeltaTable)
    try:
        todos = list(file_system_squad1.get_paths(path=pasta_silver, recursive=True))
        pastas_particao = sorted(set(
            "/".join(
                parte for parte in item.name.replace(pasta_silver + "/", "").split("/")
                if "=" in parte
            )
            for item in todos
            if item.is_directory and "=" in item.name
        ))

        if pastas_particao:
            for p in pastas_particao:
                print(p)
            print(f"\nTotal de partições físicas: {len(pastas_particao)}")
        else:
            print("Nenhuma partição física encontrada.")
    except Exception as e:
        print("Não foi possível listar partições:", e)
else:
    print("Atenção: a tabela Delta da Silver ainda não foi encontrada no caminho físico esperado.")

In [0]:
# ==========================================================
# CATÁLOGO DE TABELAS DELTA NAS CAMADAS SILVER
# squad1/silver/*
# squad2/silver/*
# squad3/silver/*
# ==========================================================

from deltalake import DeltaTable
import pandas as pd

containers = ["squad1", "squad2", "squad3"]

resultado = []

for container in containers:

    print(f"\nAnalisando container: {container}")

    try:

        fs = service_client.get_file_system_client(container)

        caminhos = list(fs.get_paths(path="silver"))

        # Encontrar todas as pastas que possuem _delta_log
        tabelas_delta = set()

        for item in caminhos:

            if "_delta_log" in item.name:

                tabela_path = item.name.split("/_delta_log")[0]

                tabelas_delta.add(tabela_path)

        print(f"Tabelas encontradas: {len(tabelas_delta)}")

        for tabela_path in sorted(tabelas_delta):

            try:

                uri = (
                    f"abfss://{container}"
                    f"@{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/"
                    f"{tabela_path}"
                )

                dt = DeltaTable(
                    uri,
                    storage_options=STORAGE_OPTIONS
                )

                schema = dt.schema()

                for campo in schema.fields:

                    resultado.append({
                        "container": container,
                        "tabela": tabela_path.split("/")[-1],
                        "caminho_fisico": tabela_path,
                        "coluna": campo.name,
                        "tipo": str(campo.type)
                    })

            except Exception as e:

                print(
                    f"Erro ao ler schema da tabela "
                    f"{tabela_path}: {e}"
                )

    except Exception as e:

        print(
            f"Erro ao acessar container "
            f"{container}: {e}"
        )

# ==========================================================
# RESULTADO FINAL
# ==========================================================

df_schema = pd.DataFrame(resultado)

if len(df_schema) == 0:
    print("Nenhuma tabela Delta encontrada.")
else:
    display(
        df_schema.sort_values(
            ["container", "tabela", "coluna"]
        )
    )

In [0]:
for (container, tabela), grupo in df_schema.groupby(
    ["container", "tabela"]
):

    print("\n" + "="*80)
    print(f"Container : {container}")
    print(f"Tabela    : {tabela}")
    print("="*80)

    display(
        grupo[
            ["coluna", "tipo"]
        ].reset_index(drop=True)
    )